In [131]:
import pandas as pd


#                     Part A — Diagnose the Raw File
# 1. Load the dataset and inspect head(), dtypes, and info().
df = pd.read_csv("case2_online_orders_raw.csv")
# print(df.head())
print(df.dtypes) #->All are object
print(df.info()) #->In quantity one value is not null


# 2. List the columns that should be numeric, datetime, Boolean, or category.

# numeric -> quantity, unit_price, _rating,
# datetime -> order_time
# boolean -> priority
# category -> status, category,


# 3. Find examples of values that will fail if converted directly.
# priority-> (TRUE, FALSE, Y, N, YES, ),
# unit_price-> (1,499.50,  8,750, 1,250),
# quantity-> (2 pcs, one ,NaN),
# rating-> (Not Rated),
# orderdate ->  (bad date)



Order_ID      object
Product       object
Quantity      object
Unit_Price    object
Order_Time    object
Status        object
Priority      object
Category      object
Rating        object
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Order_ID    12 non-null     object
 1   Product     12 non-null     object
 2   Quantity    11 non-null     object
 3   Unit_Price  12 non-null     object
 4   Order_Time  12 non-null     object
 5   Status      12 non-null     object
 6   Priority    12 non-null     object
 7   Category    12 non-null     object
 8   Rating      12 non-null     object
dtypes: object(9)
memory usage: 996.0+ bytes
None


In [132]:

#                     Part B — Product and Numeric Fields
# 1. Create df_clean as a copy and remove leading/trailing spaces from Product.
df_clean = df.copy()
df_clean["Product"] = df_clean["Product"].str.strip()

for  val in df_clean["Product"]:
  print(repr(val))



'Laptop Stand'
'Wireless Mouse'
'Office Chair'
'Notebook Pack'
'USB Hub'
'Desk Lamp'
'Pen Set'
'Bookshelf'
'Webcam'
'Marker Pack'
'Router'
'Storage Cabinet'


In [133]:
# 2. Convert Quantity to nullable Int64 using safe numeric conversion.
df_clean["Quantity"] = pd.to_numeric(df_clean["Quantity"],errors="coerce").astype("Int64")


# 3. Identify the Quantity values that became missing after conversion
df_clean[df_clean["Quantity"].isna()]



,Order_ID,Product,Quantity,Unit_Price,Order_Time,Status,Priority,Category,Rating
2,ORD003,Office Chair,<NA>,6500,06-07-2026 09:00,Shipped,YES,Furniture,4.2
5,ORD006,Desk Lamp,<NA>,1800,07-07-2026 16:20,Cancelled,No,Furniture,3.9
8,ORD009,Webcam,<NA>,3200,09/07/2026 10:05,Delivered,Yes,Electronics,4.8


In [134]:
# 4. Clean commas and spaces from Unit_Price and convert it to numeric.
df_clean["Unit_Price"] =df_clean["Unit_Price"].str.strip()
df_clean["Unit_Price"] = df_clean["Unit_Price"].str.replace(",", "", regex=False)
df_clean["Unit_Price"] = df_clean["Unit_Price"].astype("float64")
df_clean["Unit_Price"]

0     1499.5
1      799.0
2     6500.0
3      350.0
4     1250.0
5     1800.0
6      250.0
7     8750.0
8     3200.0
9      300.0
10    2600.0
11    9500.0
Name: Unit_Price, dtype: float64

In [135]:
# 5. Explain why '1,499.50' cannot be handled correctly by a simple astype(float) before cleaning.

# "1,499.50" contains a comma, so it is not in a valid numeric format for astype(float). We remove the comma firstand then convert it to float.

In [136]:
# Part C — Date/Time and Rating
# 1. Convert Order_Time to datetime while allowing mixed day-first formats.
df_clean["Order_Time"] = pd.to_datetime(df_clean["Order_Time"], errors='coerce', format='mixed', dayfirst=True)
# 2. Identify the invalid Order_Time that becomes NaT.
df_clean["Order_Time"].isna()
# 3. Convert Rating to numeric using errors='coerce'.
df_clean["Rating"] = pd.to_numeric(df_clean["Rating"], errors='coerce')
# 4. Identify the rating value that becomes NaN.
df_clean["Rating"].isna()



0     False
1     False
2     False
3      True
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
Name: Rating, dtype: bool

In [137]:
# Part D — Boolean and Category
# 1. Standardize Priority values such as Yes, Y, TRUE, No, N, and FALSE and convert them to nullable boolean.
df_clean["Priority"] = df_clean["Priority"].replace({
    "YES" : "Yes",
    "N": "No",
    "Y" : "Yes",
    "FALSE" : "No",
    "TRUE": "Yes"
})
df_clean["Priority"] = df_clean["Priority"].replace({"Yes" :True, "No":False}
).astype("boolean")

df_clean["Priority"]

# 2. Convert Status to category.
df_clean["Status"] = df_clean["Status"].astype("category")
df_clean["Status"]

# 3. Convert Category to category.
df_clean["Category"] = df_clean["Category"].astype("category")
df_clean["Category"]

# 4. Display the category labels for Status and Category.
print(df_clean["Category"].cat.categories)
print(df_clean["Status"].cat.categories)



Index(['Electronics', 'Furniture', 'Stationery'], dtype='object')
Index(['Cancelled', 'Delivered', 'Pending', 'Shipped'], dtype='object')


C:\Users\KARAN\AppData\Local\Temp\ipykernel_23032\3736053878.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_clean["Priority"] = df_clean["Priority"].replace({"Yes" :True, "No":False}


In [ ]:
# Part E — Final Verification
# 1. Display all final dtypes.
print(df_clean.dtypes)
# 2. Count conversion-created missing values in Quantity, Order_Time, and Rating.
df_clean[["Quantity", "Order_Time", "Rating"]].isna().sum()

# 3. Display rows where at least one of those three fields is missing.
df_clean[df_clean["Quantity"].isna() | df_clean["Order_Time"].isna() | df_clean["Rating"].isna() ]

# 4. Explain the difference between NaN, NaT, and <NA> in the final result.
# NaN  -> Missing value in numeric data
# NaT  -> Missing/invalid datetime value
# <NA> -> Pandas missing value used by nullable dtypes /Not Availiable

# 5. Save your converted dataset and compare it with the supplied clean dataset.
print(df)
print(df_clean)


   Order_ID          Product Quantity Unit_Price        Order_Time     Status  \
0    ORD001    Laptop Stand         2   1,499.50  05-07-2026 10:30  Delivered   
1    ORD002   Wireless Mouse        3        799  05/07/2026 11:15  Delivered   
2    ORD003     Office Chair      one       6500  06-07-2026 09:00    Shipped   
3    ORD004    Notebook Pack        5       350   06/07/2026 14:45    Pending   
4    ORD005          USB Hub        2      1,250          bad date  Delivered   
5    ORD006        Desk Lamp      NaN       1800  07-07-2026 16:20  Cancelled   
6    ORD007          Pen Set       10        250  08/07/2026 12:00  Delivered   
7    ORD008        Bookshelf        1      8,750  08-07-2026 17:10    Shipped   
8    ORD009           Webcam    2 pcs       3200  09/07/2026 10:05  Delivered   
9    ORD010      Marker Pack        6        300  09-07-2026 13:25    Pending   
10   ORD011           Router        2       2600  10/07/2026 09:40  Delivered   
11   ORD012  Storage Cabinet

In [ ]:
#  Part F — Short Interpretation Questions
# 1. Why is errors='coerce' safer than forcing a conversion on messy imported data?
# errors = coerce will simply convert into NaN / NaT if the value is incorrect/missing

# 2. Why should commas be removed from price strings before numeric conversion?
# because commas is not a good/ standard format for numeric conversion, thats why we convert "1,500" to "1500" then it will be easy to convert


# 3. Why are Status and Category good candidates for category dtype?
# because we can use groupby and differentiate rows on the category dtype


# 4. Why is Order_ID better kept as text rather than converted to category or number?
# becase it contains 'ORD' thats why not numeric AND ID are differnt so we cant categorise rows upon Order ID